# SceneProof — GPU 한 장으로 30fps 카메라 몇 대까지? (Google Colab)

**무엇을 하나요?**
1. 30fps 샘플 영상(1280×720)을 카메라 N대인 것처럼 **실시간으로** 한 분석기에 넣고, N을 1, 2, 3 … 으로 늘려 가며 **모든 카메라가 30fps의 98% 이상을 분석하는지** 잽니다. 사람 찾기는 TensorRT FP16(GPU)입니다. 기준은 실행 전에 `configs/bench/gpu_channels.yaml`에 적어 두었습니다.
2. 같은 측정을 설정만 바꿔 두 번 더 합니다: 근거 장면을 **1초에 10장만** 저장(`ring_fps: 10`), 그리고 근거 클립 인코더를 **1차 실행 때 방식**(보통 우선순위, 모든 코어)으로 되돌린 것. 1차 실행에서 프레임이 클립을 인코딩하는 동안 버려진다는 것을 찾아 인코더를 낮은 우선순위로 바꿨고, 이번 실행은 그 전후를 같은 자리에서 비교합니다.
3. 확인 두 가지: (a) 영상 4편을 **동시에** 분석해도 한 편씩 분석한 것과 사건이 같은가, (b) 근거 장면을 덜 저장해도 사건과 근거 사진·클립이 그대로인가.

**하는 법**
1. 메뉴 **런타임 → 런타임 유형 변경 → `T4 GPU`** → 저장
2. 메뉴 **런타임 → 모두 실행**
3. 끝나면 `gpu_channels_results.zip`이 내려받아집니다. 그 파일을 Claude에게 올려 주세요.

걸리는 시간: 약 25분(설치 3~5분, TensorRT 엔진 만들기 약 3분, 측정 약 17분). 중간에 끊기면 다시 **모두 실행**하면 됩니다.

## 1. GPU 확인

In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout or r.stderr)
if r.returncode != 0:
    raise SystemExit("GPU가 없습니다. 메뉴 '런타임 → 런타임 유형 변경'에서 'T4 GPU'를 고르고 저장한 뒤 다시 '모두 실행'하세요.")

## 2. 코드 받기

In [ ]:
%cd /content
!rm -rf sceneproof && git clone -q --depth 1 https://github.com/sokldjs554/sceneproof.git
%cd /content/sceneproof
!git log -1 --format='코드 버전: %h %s'

## 3. 설치 (3~5분)
Colab에 이미 있는 PyTorch는 그대로 쓰고, ONNX Runtime GPU판과 TensorRT를 추가로 설치합니다. 빨간 의존성 경고가 나와도 다음 칸이 실행되면 괜찮습니다.

In [ ]:
!pip -q install -e . --no-deps
!pip -q install "pydantic-settings>=2.4" "grpcio>=1.66" "protobuf>=7.35.1" psutil onnx onnxconverter-common
!pip -q uninstall -y onnxruntime
!pip -q install onnxruntime-gpu tensorrt-cu12
!python -c "import onnxruntime as o, tensorrt as t, torch; print('onnxruntime', o.__version__, o.get_available_providers()); print('tensorrt', t.__version__); print('torch', torch.__version__, 'cuda', torch.version.cuda, torch.cuda.get_device_name(0))"
!nproc; ffmpeg -version | head -1

## 4. 모델과 영상 받기

In [ ]:
!ONNX_FROM=official VARIANTS="yolox_tiny" bash scripts/fetch_models.sh
!bash scripts/fetch_data.sh
!mkdir -p results

## 5. 측정 (약 17~20분)
처음에 TensorRT 엔진을 만드느라 2~3분 동안 아무것도 안 나옵니다. 그 뒤 채널 수마다 한 줄씩 나옵니다(`PASS` = 모든 카메라가 30fps의 98% 이상 분석).

In [ ]:
!python scripts/gpu_channels.py --out results/gpu_channels.json 2>&1 | tee results/gpu_channels.log

## 6. 결과 요약

In [ ]:
import json
d = json.load(open("results/gpu_channels.json"))
print("코드", d["git"], "|", d["env"].get("gpu"), "|", d["env"].get("cpu"), d["env"].get("cpu_count"), "코어 |", "계획에서 바꾼 것:", d["overrides"] or "없음")
for s in d["sweeps"]:
    name = f"{s['label']} / {s['variant']}"
    if s["status"] != "ok":
        print(f"\n[{name}] 실패: {s['error'][:300]}"); continue
    print(f"\n[{name}] 30fps를 유지한 최대 채널 수: {s['capacity_channels']}  (처음 실패한 채널 수: {s['first_failing_channels']}, 클립 인코더: {s.get('clip_encoder')})")
    print(f"{'채널':>4} {'결과':>5} {'가장 느린 채널 fps':>16} {'버린 프레임':>10} {'지연 p95(ms)':>12} {'CPU%':>6} {'GPU%':>6}   프레임당 ms: 근거저장/전처리/추론/후처리/추적/구역")
    for r in s["rows"]:
        st = r["stage_ms_mean"]; g = r["gpu"]["util_percent_mean"] if r["gpu"] else float("nan")
        print(f"{r['channels']:>4} {'통과' if r['pass'] else '실패':>5} {r['processed_ratio_min']*d['plan']['target_fps']:>16.1f} {r['dropped_total']:>10} {r['lag_ms_p95_worst']:>12.0f} {r['cpu_percent_system']:>6.0f} {g:>6.0f}   "
              f"{st['evidence_encode']:.1f} / {st['preprocess']:.1f} / {st['infer']:.1f} / {st['postprocess']:.1f} / {st['track']:.1f} / {st['zones']:.1f}")
iso, evc = d.get("isolation"), d.get("evidence_check")
if iso and iso.get("status") == "ok":
    print("\n동시에 분석 vs 한 편씩:", "사건이 모두 같음" if iso["all_same"] else "다름!", {v: (c["events_together"], c["events_alone"]) for v, c in iso["videos"].items()})
elif iso:
    print("\n동시에 분석 vs 한 편씩:", iso)
if evc and evc.get("status") == "ok":
    print("근거 장면 덜 저장:", "사건이 모두 같음" if evc["all_same_events"] else "사건이 달라짐!")
    for v, c in evc["videos"].items():
        a, b = c["every_frame"], c[f"ring_fps_{evc['ring_fps']:g}"]
        print(f"  {v}: 근거 저장 {a['evidence_encode_ms_mean']} → {b['evidence_encode_ms_mean']} ms/프레임, 클립 {a['evidence']['clips']}개({a['evidence']['clip_frames']}장) → {b['evidence']['clips']}개({b['evidence']['clip_frames']}장), "
              f"진입 사진 시각 오차 최대 {b['evidence']['entry_picture_offset_s_max']}초, 이탈 사진 {b['evidence']['exit_picture_offset_s_max']}초")
elif evc:
    print("근거 장면 덜 저장:", evc)

## 7. 결과 내려받기
`gpu_channels_results.zip`을 Claude에게 올려 주세요. 영상은 들어 있지 않고 숫자(JSON)와 기록만 들어 있습니다.

In [ ]:
!cd results && zip -q -r ../gpu_channels_results.zip gpu_channels.json gpu_channels.log && cd .. && ls -la gpu_channels_results.zip
try:
    from google.colab import files
    files.download("gpu_channels_results.zip")
except Exception as e:
    print("자동 다운로드가 안 되면 왼쪽 파일 창에서 sceneproof/gpu_channels_results.zip을 내려받으세요.", e)